In [ ]:
import sys,subprocess,os,time
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['HF_HUB_DISABLE_XET']='1'
print('INSTALL_BEGIN',flush=True)
subprocess.run([sys.executable,'-m','pip','install','-q',
 'diffusers>=0.35.0,<0.42.0','transformers>=4.47.0,<4.52.0',
 'accelerate','sentencepiece','imageio[ffmpeg]','safetensors','pillow','numpy'],
 check=True)
import torch, diffusers
print('INSTALL_COMPLETE','torch',torch.__version__,'diffusers',diffusers.__version__,flush=True)
print('GPU',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT_AVAILABLE',flush=True)
assert torch.cuda.is_available(),'Kaggle accelerator required'


In [ ]:
from pathlib import Path
from PIL import Image, ImageOps
import json, numpy as np, torch, gc, traceback, os, subprocess, time, math
from diffusers import LTXConditionPipeline
from diffusers.utils import export_to_video
ASSETS = list(Path('/kaggle/input').rglob('speech_00.wav'))
assert ASSETS,'Missing private VoxCPM2 assets dataset'
SRC=ASSETS[0].parent
IMAGES=[SRC / ('scene_%02d.jpg'%i) for i in range(8)]
assert all(p.exists() for p in IMAGES),[(p.name,p.exists()) for p in IMAGES]
SCENES=json.loads((SRC/'script.json').read_text())
assert len(SCENES)==8
OUT=Path('/kaggle/working/ltx_death_documentary');OUT.mkdir(exist_ok=True)
print('INPUTS_OK',SRC,'narration', (SRC/'narration.wav').stat().st_size,flush=True)
MOTION=[
 'Cinematic night sky with subtle drifting clouds and twinkling stars, distant silhouetted figure stays still, gentle slow dolly forward, stable documentary motion, no text.',
 'Cinematic close view of a symbolic anatomical heart model, soft red light pulses gently around the model, slow cinematic camera orbit, stable composition, educational documentary, no text.',
 'Scientific visualization of interconnected neurons, glowing electric blue pulses move along pathways then slowly fade, subtle light transitions, stable and precise, no text.',
 'Empty hospital corridor, camera slowly moves forward, soft morning light changes gently, subtle airborne dust motes, no people, cinematic documentary, no text.',
 'Nature scene with fallen autumn leaves, delicate mist slowly drifts and leaves flutter in a light breeze, calm photographic realism, no text.',
 'Beautiful deep indigo celestial stars, slow drifting distant particles and very subtle cinematic zoom toward a warm distant light, no text.',
 'Abstract transparent human profile silhouette with gentle neural shimmer and subtle floating particles, static composition with organic luminous movement, no text.',
 'Tranquil ocean sunrise, golden clouds slowly shift, gentle wave movement and expanding warm sunrise, peaceful cinematic finale, no text.'
]


In [ ]:
# Prefer the 2B 0.9.8 distilled checkpoint. Never silently label a fallback "distilled".
MODEL_USED='Lightricks/ltxv-2b-0.9.8-distilled.safetensors'
MODEL_ERROR=None
pipe=None
torch.cuda.empty_cache()
try:
    from diffusers import LTXVideoTransformer3DModel
    print('TRY_DISTILLED_2B',flush=True)
    from huggingface_hub import hf_hub_download
    checkpoint=hf_hub_download(repo_id='Lightricks/LTX-Video',filename='ltxv-2b-0.9.8-distilled.safetensors')
    transformer=LTXVideoTransformer3DModel.from_single_file(checkpoint,torch_dtype=torch.float16)
    pipe=LTXConditionPipeline.from_pretrained('Lightricks/LTX-Video-0.9.5',
      transformer=transformer,torch_dtype=torch.float16)
    pipe.enable_model_cpu_offload()
    if hasattr(pipe.vae,'enable_tiling'):pipe.vae.enable_tiling()
    print('DISTILLED_2B_LOADED',flush=True)
except Exception as e:
    MODEL_ERROR=str(e)[:500]
    print('DISTILLED_2B_LOAD_FAILED',MODEL_ERROR,flush=True)
    for key in ['transformer','pipe']:
        if key in globals():globals()[key]=None
    gc.collect();torch.cuda.empty_cache()
    print('FALLBACK_STABLE_2B_0.9.5',flush=True)
    pipe=LTXConditionPipeline.from_pretrained('Lightricks/LTX-Video-0.9.5',
      torch_dtype=torch.float16)
    pipe.enable_model_cpu_offload()
    if hasattr(pipe.vae,'enable_tiling'):pipe.vae.enable_tiling()
    MODEL_USED='Lightricks/LTX-Video-0.9.5 (2B non-distilled fallback)'
print('MODEL_USED',MODEL_USED,flush=True)


In [ ]:
from diffusers.utils import export_to_video
results=[]
FPS=24
NEG='flickering objects, glitches, artificial transformations, deformation, low quality, jittery, unreadable captions, watermark'
for i,src_image in enumerate(IMAGES):
    target=OUT / ('motion_%02d.mp4'%i)
    if target.exists() and target.stat().st_size>20000:
        results.append({'scene':i,'status':'ai','path':str(target),'bytes':target.stat().st_size})
        print('REUSED_AI_CLIP',i,flush=True)
        continue
    image=ImageOps.fit(Image.open(src_image).convert('RGB'),(512,512),method=Image.Resampling.LANCZOS)
    image.save(OUT/('input_%02d.jpg'%i))
    prompt=MOTION[i]
    print('I2V_START',i+1,'/8',MODEL_USED,flush=True)
    try:
        import inspect
        kwargs=dict(image=image,prompt=prompt,negative_prompt=NEG,width=512,height=512,
           num_frames=49,num_inference_steps=8 if 'distill' in MODEL_USED.lower() and 'non-distilled' not in MODEL_USED.lower() else 28,
           guidance_scale=1.0 if 'distill' in MODEL_USED.lower() and 'non-distilled' not in MODEL_USED.lower() else 4.5,
           decode_timestep=0.05,decode_noise_scale=0.025,
           generator=torch.Generator('cpu').manual_seed(20261103+i))
        frames=pipe(**kwargs).frames[0]
        export_to_video(frames,str(target),fps=FPS)
        assert target.exists() and target.stat().st_size>10000
        results.append({'scene':i,'status':'ai','path':str(target),'bytes':target.stat().st_size})
        print('I2V_SUCCESS',i+1,target.stat().st_size,flush=True)
        del frames;gc.collect();torch.cuda.empty_cache()
    except Exception as exc:
        print('I2V_SCENE_FAILED',i+1,repr(exc)[:550],flush=True)
        results.append({'scene':i,'status':'fallback','path':None,'error':str(exc)[:350]})
        gc.collect();torch.cuda.empty_cache()
    (OUT/'generation_report.json').write_text(json.dumps({'model':MODEL_USED,'model_load_error':MODEL_ERROR,'scenes':results},indent=2))
print('I2V_RESULTS',json.dumps({'ai':sum(x['status']=='ai' for x in results),'fallback':sum(x['status']!='ai' for x in results)}),flush=True)
